# 2. Gravação EEG para DataLoader do PyTorch

Um modelo treina em tensor batches, não em traços contínuos de voltagem. Precisamos dividir a gravação EEG longa e contínua em vários tensores de tamanhos iguais para serem processados pela rede.

## 2.1 Setup

In [21]:
import braindecode
from braindecode.preprocessing import (
    Preprocessor,
    create_fixed_length_windows,
    create_windows_from_events,
    preprocess,
)

import torch
from torch.utils.data import DataLoader
from eegdash.dataset import DS002721
import pandas as pd
import numpy as np

## 2.2 Dataset vs Dataloader

A classe `torch.utils.data.Dataset` é para armazenamento e recuperação de dados. A classe `torch.utils.data.DataLoader` consome o dataset e adiciona: 

- **batching (Agrupamento)** - Agrupa múltiplos exemplos do dataset em lotes (batches) para processamento paralelo.

- **shuffling** - Mistura a ordem dos dados antes de dividi-los em lotes a cada época de treinamento. Evita que a rede decore sequências.

- **worker orchestration** - Coordenação de múltiplos processos para ler do disco, decodificar e aplicar transformações nos dados em paralelo. Previne gargalos de I/O na CPU.

Mas não armazena dados. É um iterável que, por trás, chama `__getitem__` no dataset e empilha os resultados.

```text
EEGDashDataset                  WindowsDataset                DataLoader
(records + BIDS meta)           (cut samples for the model)   (consumer)
┌────────────────────┐          ┌───────────────────┐         ┌────────────┐
│ record 0 (Raw) ──┐ │ preproc  │ window 0 (X, y)   │ batch + │ batch 0    │
│ record 1 (Raw) ──┼─┼─────────▶│ window 1 (X, y)   │ shuffle │ batch 1    │
│ record 2 (Raw) ──┘ │ + cut    │ ...               │────────▶│ ...        │
│ ...                │ windows  │ window N (X, y)   │         │ batch K    │
└────────────────────┘          └───────────────────┘         └────────────┘
  __len__  = n_records           __len__  = n_windows           iter() yields
  __getitem__ -> (raw, ...)       __getitem__ -> (X, y, idx)      stacked tensors
```

## 2.3 Corolários

- **Uma amostra é uma janela, não uma gravação inteira** - Um EEG contínuo é um array longo por sessão. Modelos treinam em frames de tamanho-fixo, então o pipeline corta cada `raw` em tensores `(n_channels, window_samples)` antes de qualquer batch acontecer.

- **Dois tipos de janelas** - `brandecode.preprocessing.create_fixed_lenght_windows` avança através do sinal contínuo, ignorando eventos, útil para pré-treino auto-supervisionado e estágio de sono. `braindecode.preprocessing.create_windows_from_events` corta em volta de eventos BIDS com offsets explícitos. Essa é a escolha certa para ERP e tarefas relacionadas.

- **Acesso aleatório vs armazenamento sequencial** - O treino embaralha as janelas. Para selecionar um `X[i]` aleatório, o custo de acesso cresce linearmente com o tamanho do arquivo. Zarr armazena blocos de tamanho-fixo e lê qualquer janela em dezenas de milissegundos.

## 2.4 Pegando dataset

In [22]:
dataset = DS002721(cache_dir="./data", subject="01")
dataset

[09/13/26 12:53:07] WARNING  MNE-BIDS failed due to missing file (File does not exist:                 ]8;id=9549277;file:///home/arielalves/Desktop/Iniciacao-cientifica-EEG-FM/EEGDash-Tutorials/Translated-Tutorials/.venv/lib/python3.12/site-packages/eegdash/dataset/base.py\base.py]8;;\:]8;id=9549278;file:///home/arielalves/Desktop/Iniciacao-cientifica-EEG-FM/EEGDash-Tutorials/Translated-Tutorials/.venv/lib/python3.12/site-packages/eegdash/dataset/base.py#1848\1848]8;;\
                             data/ds002721/sub-01/eeg/sub-01_task-_eeg.edf                                         
                             Did you mean one of:                                                                  
                             sub-01_task-run6_eeg.edf                                                              
                             sub-01_task-run5_eeg.edf                                                              
                             sub-01_task-run4_eeg.edf                                                              
                             instead of:                                                                           
                             sub-01_task-_eeg.edf), falling back to direct MNE reader.                             

                    WARNING  Falling back to direct .edf reader for sub-01_task-run3_eeg.edf (bypassing  ]8;id=9549283;file:///home/arielalves/Desktop/Iniciacao-cientifica-EEG-FM/EEGDash-Tutorials/Translated-Tutorials/.venv/lib/python3.12/site-packages/eegdash/dataset/io.py\io.py]8;;\:]8;id=9549284;file:///home/arielalves/Desktop/Iniciacao-cientifica-EEG-FM/EEGDash-Tutorials/Translated-Tutorials/.venv/lib/python3.12/site-packages/eegdash/dataset/io.py#1652\1652]8;;\
                             MNE-BIDS).                                                                            

<BaseConcatDataset | 6 EEGDashRaw(s) | 2791000 total samples>
  Sfreq*: 1000.0 Hz
  Channels*: 19 (19 EEG)
  Ch. names*: FP1, FP2, F7, F3, Fz, F4, F8, T3, C3, Cz, ... (+9 more)
  Duration*: 552.0 s
  (* from first recording)
  Description: 6 recordings × 7 columns [subject, task, age, sex, session, run, gender]

Aqui, **Total Samples** significa o número total de pontos no tempo coletados.

In [23]:
record = dataset.datasets[0]
raw = record.raw

raw

<RawEDF | sub-01_task-run3_eeg.edf, 19 x 552000 (552.0 s), ~19 KiB, data not loaded>

In [24]:
n_channels = len(raw.ch_names)
n_channels

19

In [25]:
sfreq = float(raw.info["sfreq"])
sfreq

1000.0

## 2.5 Dois pré-processamentos seguros

1. Como `raw.info['sfreq']` vai ficar após um resample de 100Hz (intervalo entre pontos mudar de 1ms para 10ms)?
2. Como `len(raw.ch_names)` vai ficar após remover os canais que não são EEG?

`pick_types(eeg=True)` mantém apenas EEG, e `resample(sfreq=100)` faz o resampling.

In [26]:
TARGET_SFREQ = 100  # Hz

preprocess(
    dataset,
    [
        Preprocessor("pick_types", eeg=True, eog=False, misc=False), #Mantém apenas canais EEG
        Preprocessor("resample", sfreq=TARGET_SFREQ), #Muda amostragem para 100Hz
    ],
)

raw = dataset.datasets[0].raw
n_channels = len(raw.ch_names)
sfreq = float(raw.info["sfreq"])

# imprime tudo em um dataset pandas
pd.Series(
    {
        "n_channels": n_channels,
        "sfreq (Hz)": sfreq,
        "dtype": str(raw.get_data().dtype),
    },
    name="value",
).to_frame()

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/home/arielalves/Desktop/Iniciacao-cientifica-EEG-FM/EEGDash-Tutorials/Translated-Tutorials/.venv/lib/python3.12/site-packages/braindecode/preprocessing/preprocess.py:78: UserWarning: apply_on_array can only be True if fn is a callable function. Automatically correcting to apply_on_array=False.
  warn(


[09/13/26 12:53:10] WARNING  MNE-BIDS failed due to missing file (File does not exist:                 ]8;id=9549289;file:///home/arielalves/Desktop/Iniciacao-cientifica-EEG-FM/EEGDash-Tutorials/Translated-Tutorials/.venv/lib/python3.12/site-packages/eegdash/dataset/base.py\base.py]8;;\:]8;id=9549290;file:///home/arielalves/Desktop/Iniciacao-cientifica-EEG-FM/EEGDash-Tutorials/Translated-Tutorials/.venv/lib/python3.12/site-packages/eegdash/dataset/base.py#1848\1848]8;;\
                             data/ds002721/sub-01/eeg/sub-01_task-_eeg.edf                                         
                             Did you mean one of:                                                                  
                             sub-01_task-run6_eeg.edf                                                              
                             sub-01_task-run5_eeg.edf                                                              
                             sub-01_task-run4_eeg.edf                                                              
                             instead of:                                                                           
                             sub-01_task-_eeg.edf), falling back to direct MNE reader.                             

                    WARNING  Falling back to direct .edf reader for sub-01_task-run2_eeg.edf (bypassing  ]8;id=9549295;file:///home/arielalves/Desktop/Iniciacao-cientifica-EEG-FM/EEGDash-Tutorials/Translated-Tutorials/.venv/lib/python3.12/site-packages/eegdash/dataset/io.py\io.py]8;;\:]8;id=9549296;file:///home/arielalves/Desktop/Iniciacao-cientifica-EEG-FM/EEGDash-Tutorials/Translated-Tutorials/.venv/lib/python3.12/site-packages/eegdash/dataset/io.py#1652\1652]8;;\
                             MNE-BIDS).                                                                            

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


...

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


[09/13/26 12:53:21] WARNING  MNE-BIDS failed due to missing file (File does not exist:                 ]8;id=9549337;file:///home/arielalves/Desktop/Iniciacao-cientifica-EEG-FM/EEGDash-Tutorials/Translated-Tutorials/.venv/lib/python3.12/site-packages/eegdash/dataset/base.py\base.py]8;;\:]8;id=9549338;file:///home/arielalves/Desktop/Iniciacao-cientifica-EEG-FM/EEGDash-Tutorials/Translated-Tutorials/.venv/lib/python3.12/site-packages/eegdash/dataset/base.py#1848\1848]8;;\
                             data/ds002721/sub-01/eeg/sub-01_task-_eeg.edf                                         
                             Did you mean one of:                                                                  
                             sub-01_task-run6_eeg.edf                                                              
                             sub-01_task-run5_eeg.edf                                                              
                             sub-01_task-run4_eeg.edf                                                              
                             instead of:                                                                           
                             sub-01_task-_eeg.edf), falling back to direct MNE reader.                             

                    WARNING  Falling back to direct .edf reader for sub-01_task-run6_eeg.edf (bypassing  ]8;id=9549343;file:///home/arielalves/Desktop/Iniciacao-cientifica-EEG-FM/EEGDash-Tutorials/Translated-Tutorials/.venv/lib/python3.12/site-packages/eegdash/dataset/io.py\io.py]8;;\:]8;id=9549344;file:///home/arielalves/Desktop/Iniciacao-cientifica-EEG-FM/EEGDash-Tutorials/Translated-Tutorials/.venv/lib/python3.12/site-packages/eegdash/dataset/io.py#1652\1652]8;;\
                             MNE-BIDS).                                                                            

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


,value
n_channels,19
sfreq (Hz),100.0
dtype,float64


Temos 19 canais.

## 2.6 Cortar em janelas de tamanho-fixo

- `window_size_samples = int(WINDOW_SECONDS * TARGET_SFREQ)` **(Tamanho da Janela)** - Quantidade de pontos em cada janela. Para ter uma janela de 2 segundos gravando a 100 Hz (100 pontos por segundo), então uma só amostra/janela deverá terá 200 pontos.

- `window_stride_samples` **(Passo/Stride)**- Quantos pontos a janela pula para frente antes de cortar a próxima.
    - _Passo = Tamanho_ - Janelas consecutivas sem sobreposição (0% overlap)
    - _Passo menor_ - Cria sobreposição (overlap). Isso gera mais janelas, mas elas compartilham dados.

Diferença entre size e stride:
- Size = 200, stride = 100 -> Anda 100, pega 200 (com sobreposição)
- Size = 200, stride = 200 -> Anda 200, pega 200 (0% sobreposição)
- Size = 200, stride = 300 -> Anda 300, pega 200 (pula 100 pontos a cada passo)

In [27]:
WINDOW_SECONDS = 2.0
window_size_samples = int(WINDOW_SECONDS * TARGET_SFREQ)
windows = create_fixed_length_windows(
    dataset,
    window_size_samples=window_size_samples,
    window_stride_samples=window_size_samples,  # 0 % overlap
    drop_last_window=True,
    preload=True,
)
X_one, y_one, _idx = windows[0]

windows

/tmp/ipykernel_132557/2082059758.py:3: DeprecationWarning: `drop_last_window` is deprecated and will be removed in version 2.0. Use `on_last_window='drop'` if True or `on_last_window='overlap'` if False. See https://github.com/braindecode/braindecode/pull/1058 for feedback.
  windows = create_fixed_length_windows(


<BaseConcatDataset | 6 EEGWindowsDataset(s) | 1395 total samples>
  Sfreq*: 100.0 Hz
  Channels*: 19 (19 EEG)
  Ch. names*: FP1, FP2, F7, F3, Fz, F4, F8, T3, C3, Cz, ... (+9 more)
  (* from first recording)
  Description: 6 recordings × 7 columns [subject, task, age, sex, session, run, gender]
  Window: 200 samples (2.000 s)
  Targets: 1 unique ({-1: 1395})

Aqui, **Total Samples** significa o número total de janelas geradas. 1395 * 200 = 279000

Originalmente, tínhamos 2791000 a 1000Hz. Com downsampling para 100Hz, ficam 279100 pontos. Com as janelas, os últimos 100 pontos são jogados fora por não completarem a janela.

In [28]:
X_one

array([[-3.48153582e-04, -3.53973621e-04, -3.62332736e-04, ...,
        -3.12824995e-04, -2.70037126e-04, -2.40948735e-04],
       [-1.55182177e-04, -1.60386087e-04, -1.72169544e-04, ...,
        -1.24405371e-04, -8.02479262e-05, -4.58610848e-05],
       [-5.53500031e-05, -5.96929567e-05, -6.83775361e-05, ...,
         2.89998479e-06,  2.29673387e-05,  2.92474215e-05],
       ...,
       [-1.16368210e-04, -1.26511572e-04, -1.25156017e-04, ...,
        -1.14741939e-04, -1.19933640e-04, -1.19406024e-04],
       [ 2.38256271e-05,  1.01900177e-05,  5.36690459e-06, ...,
         2.24102423e-05,  1.42429490e-05,  7.77352216e-06],
       [ 3.58457473e-05,  3.08178569e-05,  2.79243050e-05, ...,
         2.73667374e-05,  1.93020951e-05,  1.79931321e-05]],
      shape=(19, 200), dtype=float32)

In [29]:
y_one

-1

In [30]:
_idx

[0, 0, 200]

Cada `windows[i]` é uma única janela.

- `windows[i][0] = X_one` = Dados do sinal `(n_canais, amostras)`
- `windows[i][1] = y_one` = Rótulo (target). O target fica marcado como `-1` se não houver rótulo específico de tarefa/evento.
- `windows[i][2] = _idx` = Lista/Tupla de índices que Braindecode usa para rastrear a origem da janela, com:
    - `_idx[0]` =  índice da gravação original de onde a janela veio
    - `_idx[1]` = Ponto de início da janela no sinal contínuo original
    - `_idx[2]` = Ponto de fim da janela no sinal contínuo original

In [31]:
# Imprime tudo em um dataset pandas
pd.Series(
    {
        "n_windows": len(windows),
        "windows[0][0].shape": str(tuple(X_one.shape)),
        "X.dtype": str(X_one.dtype),
        "window_samples": window_size_samples,
        "window_seconds": WINDOW_SECONDS,
    },
    name="value",
).to_frame()

,value
n_windows,1395
windows[0][0].shape,"(19, 200)"
X.dtype,float32
window_samples,200
window_seconds,2.0


## 2.7 Colocando em um DataLoader

Apenas quatro argumentos importam para o `DataLoader` em EEG.

- `batch_size` - De 8 a 32 é um intervalo confortável para rodar na CPU.
- `shuffle` - `True` para treinamento, `False` para avaliação.
- `num_workers` - `0` (síncrono) é o padrão correto com `preload=True`, pois as janelas já vivem na RAM. `>0` ajuda somente quando o dataset lê do disco por `_getitem_`.
- `pin_memory` - Defina para `True` se um dispositivo CUDA estiver presente e você planeja enviar batches com `.to(device, non_blocking=True)`

In [32]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
BATCH_SIZE = 8

gen = torch.Generator().manual_seed(SEED)
loader = DataLoader(
    windows,               # Janelas criadas
    batch_size=BATCH_SIZE, # Divide em lotes de tamanho BATCH_SIZE
    shuffle=True,          # Embaralhar
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    generator=gen,         # Gerador responsável pelo embaralhamento
)
X_batch, y_batch, _idx_batch = next(iter(loader))

# Imprime informações
pd.Series(
    {
        "X.shape": str(tuple(X_batch.shape)),
        "X.dtype": str(X_batch.dtype),
        "y.shape": str(tuple(y_batch.shape)),
        "y unique": str(torch.unique(y_batch).tolist()),
        "pin_memory": loader.pin_memory,
        "num_workers": loader.num_workers,
    },
    name="value",
).to_frame()

,value
X.shape,"(8, 19, 200)"
X.dtype,torch.float32
y.shape,"(8,)"
y unique,[-1]
pin_memory,False
num_workers,0


 O DataLoader é apenas o objeto principal de configuração, que guarda as regras: qual é o dataset, tamanho do batch, shuffle, etc. Ele não sabe em qual lote você está no momento.

Quando se roda `iter(loader)`, gera-se um iterador. é ele que busca os dados na memória, empilha os tensores e rastreia o índice atual do lote.

O loop for secretamente faz next(loader) a cada passo.

In [33]:
X_batch.shape

torch.Size([8, 19, 200])

In [34]:
y_batch

tensor([-1, -1, -1, -1, -1, -1, -1, -1])

In [35]:
_idx_batch

[tensor([235, 114, 192, 134,  53,  29, 101, 185]),
 tensor([47000, 22800, 38400, 26800, 10600,  5800, 20200, 37000]),
 tensor([47200, 23000, 38600, 27000, 10800,  6000, 20400, 37200])]

As janelas foram agrupadas em lotes/batches de 8.

## 2.8 Janelas Contínuas vs Event-Based Epochs

As duas funções pegam o `Raw` e levam a `(n_epochs, n_channels, n_times)`.

- `braindecode.preprocessing.create_fixed_length_windows`
    - Avança sobre o sinal contínuo e rotula toda janela com o mesmo rótulo (target) extraído dos metadados gerais daquela gravação.
    - Ideal para pré-treinamento auto-supervisionado, classificação de estágios de sono ou monitoramento contínuo onde o rótulo pertencec à sessão inteira.

- `braindecode.preprocessing.create_windows_from_events`
    - Lê anotações `mne.io.Raw.annotations` (BIDS `events.tsv` é carregado automaticamente pelo EEGDash), e define uma única janela de tamanho fixo para cada evento usando `trial_start_offset_samples`/`trial_stop_offset_samples` (você que decide), usando o código do evento como target.
    - Ideal para ERP e decodificação relacionada a eventos.
    - `trial_start_offset_samples` - Após quantos pontos do instante do evento começa a janela?
    - `trial_stop_offset_samples` - Após quantos pontos do instante do evento termina a janela?

`events.tsv` somente possui o exato instante em que ocorreu o estímulo. A janela é você quem decide.

In [36]:
# Corta janelas de 2 segundos (window_size_samples = sfreq * 2)
windows_continuous = create_fixed_length_windows(
    dataset,
    start_offset_samples=0,     # Início
    stop_offset_samples=None,   # Fim
    window_size_samples=200,    # Quantidade de pontos por janela
    window_stride_samples=200,  # Sobreposição (0%)
    drop_last_window=True,      # Deleta janela incompleta (sobra)
)

# windows_continuous[0] retorna: (X, y, window_info)
# X tem formato (19, 200) e todas as janelas dessa gravação terão o mesmo y

/tmp/ipykernel_132557/99009228.py:2: DeprecationWarning: `drop_last_window` is deprecated and will be removed in version 2.0. Use `on_last_window='drop'` if True or `on_last_window='overlap'` if False. See https://github.com/braindecode/braindecode/pull/1058 for feedback.
  windows_continuous = create_fixed_length_windows(


In [37]:
# Inspecionar anotações do primeiro registro
print(dataset.datasets[0].raw.annotations.description)

if(len(dataset.datasets[0].raw.annotations.description) != 0):
    windows_events = create_windows_from_events(
        dataset,
        trial_start_offset_samples=0,         # Início da janela a partir do instante do evento
        trial_stop_offset_samples=100,        # Fim da janela a partir do instante do evento
        mapping={"face": 0, "scrambled": 1},  # converte as anotações textuais em inteiros (se omitir, Braindecode mapeia tudo)
        preload=True
    )

# windows_events[0] retorna: (X, y, window_info)
# X tem formato (19, 100) e y varia dependendo do estímulo que disparou o corte (0 ou 1)

[]
